## Лекция 5: Автоматизация классификации и анализа судебных документов

### Цель лекции

Изучить методы автоматической классификации и анализа судебных документов с использованием машинного обучения и нейронных сетей, понять принципы применения этих технологий в юридической сфере, а также рассмотреть практические примеры с исходным кодом на языке Python.

---

### Введение

Современная правовая система генерирует колоссальные объёмы документации: судебные решения, исковые заявления, апелляционные жалобы, договоры, законодательные акты и нормативные предписания. По оценкам исследователей, только в Российской Федерации ежегодно рассматривается свыше 20 миллионов судебных дел, каждое из которых сопровождается значительным массивом текстовой документации. Ручная обработка, систематизация и анализ подобных объёмов информации представляет серьёзную практическую проблему как с точки зрения временных затрат, так и с позиции обеспечения единообразия интерпретации.

Автоматизация классификации и анализа судебных документов посредством методов машинного обучения и нейронных сетей открывает принципиально новые возможности для юридической практики. Интеллектуальные системы способны в сотни раз ускорить категоризацию документов, обеспечить единообразие применяемых критериев классификации, выявить скрытые закономерности в судебной практике и существенно снизить нагрузку на специалистов.

Вместе с тем автоматизация правовых процессов ставит ряд принципиальных вопросов: как обеспечить точность алгоритмических решений, соответствующую требованиям юридической практики? Каковы пределы применимости статистических методов к задачам интерпретации правовых норм? Как решать проблему предвзятости (bias) обученных моделей, воспроизводящих исторические паттерны судебных решений?

Настоящая лекция рассматривает технический стек методов автоматической классификации — от базовых алгоритмов машинного обучения до современных архитектур нейронных сетей — в контексте их применения к задачам обработки судебной документации. Понимание математических основ и практических аспектов каждого метода позволяет специалисту обоснованно выбирать инструментарий в зависимости от характеристик конкретной задачи.

---


### 1. Необходимые определения и теоретические основы

Прежде чем перейти к рассмотрению конкретных методов и алгоритмов, необходимо установить единый понятийный аппарат, на котором будет основываться дальнейшее изложение. Ниже приведены ключевые определения, используемые в контексте автоматизированной обработки юридических текстов.

#### 1.1. Машинное обучение

Машинное обучение (machine learning, ML) — это раздел искусственного интеллекта, занимающийся разработкой и изучением алгоритмов, которые способны обучаться на эмпирических данных и улучшать свою производительность при решении задач без явного программирования каждого шага. В отличие от детерминированных экспертных систем, методы машинного обучения извлекают правила непосредственно из данных, обнаруживая статистические закономерности, недоступные для формальной спецификации.

Формально задача обучения с учителем (supervised learning) формулируется следующим образом. Пусть задан обучающий набор данных $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^{N}$, где $x_i \in \mathbb{R}^d$ — вектор признаков $i$-го объекта, $y_i \in \mathcal{Y}$ — соответствующая метка класса, а $N$ — общее число обучающих примеров. Цель состоит в нахождении функции $f: \mathbb{R}^d \to \mathcal{Y}$, минимизирующей ожидаемую ошибку на новых, ранее не встреченных данных:

$$\mathcal{L}(f) = \mathbb{E}_{(x,y) \sim p(x,y)} [\ell(f(x), y)] \to \min_f$$

где $\ell(\hat{y}, y)$ — функция потерь, измеряющая расхождение между предсказанием $\hat{y}$ и истинным значением $y$.

В задачах классификации судебных документов $\mathcal{Y}$ представляет собой конечное множество категорий (например, гражданские, уголовные, административные, арбитражные дела), а функция потерь $\ell$ традиционно выбирается в форме категориальной перекрёстной энтропии.

#### 1.2. Нейронные сети

Искусственная нейронная сеть (artificial neural network, ANN) — это параметрическая функция $f_\theta: \mathbb{R}^d \to \mathbb{R}^K$, реализующая последовательную нелинейную трансформацию входных данных через несколько слоёв вычислительных узлов (нейронов). Каждый нейрон $j$ в слое $l$ вычисляет взвешенную сумму своих входов и применяет нелинейную функцию активации $\sigma$:

$$a_j^{(l)} = \sigma\left(\sum_{i} w_{ji}^{(l)} a_i^{(l-1)} + b_j^{(l)}\right)$$

где $w_{ji}^{(l)}$ — обучаемые веса, $b_j^{(l)}$ — смещение (bias), а $a_i^{(l-1)}$ — активации предыдущего слоя. Параметры $\theta = \{W^{(l)}, b^{(l)}\}_l$ настраиваются в процессе обучения методом обратного распространения ошибки (backpropagation).

Глубокие нейронные сети (deep neural networks) демонстрируют выдающиеся результаты в задачах обработки естественного языка благодаря способности иерархически извлекать признаки: низкоуровневые слои обнаруживают простые паттерны (символы, подслова), а высокоуровневые — семантические отношения и смысловые категории.

#### 1.3. Классификация текстов

Классификация текстов (text classification) — задача автоматического присвоения текстовым документам одной или нескольких заранее определённых категорий. Формально, при бинарной классификации $\mathcal{Y} = \{0, 1\}$, а при многоклассовой — $\mathcal{Y} = \{1, 2, \ldots, K\}$ с $K \geq 3$ классами. Задача многоклассовой классификации судебных документов является частным случаем многоклассовой классификации с взаимно исключающими категориями.

#### 1.4. Обнаружение аномалий

Обнаружение аномалий (anomaly detection) — задача идентификации объектов, чьи характеристики значительно отличаются от типичных представителей обучающего распределения $p(x)$. В контексте судебной аналитики аномальные документы могут свидетельствовать о нетипичных формулировках решений, нарушениях устоявшейся судебной практики или потенциальных признаках коррупционной деятельности.

---


### 2. Автоматическая классификация судебных документов

#### 2.1. Постановка задачи классификации

Классификация судебных документов является частным случаем задачи классификации текстов и обладает рядом специфических особенностей, принципиально отличающих её от классических бенчмарков обработки естественного языка. Юридические тексты характеризуются высокой терминологической специализированностью, сложными синтаксическими конструкциями, устойчивыми формульными оборотами и семантической многозначностью ряда ключевых понятий.

В зависимости от уровня детализации классификации судебных документов можно выделить несколько иерархических уровней:

На первом, наиболее грубом уровне документы делятся по отраслевой принадлежности: гражданское, уголовное, административное и арбитражное судопроизводство. На втором уровне внутри каждой отрасли выделяются подкатегории — например, в гражданском праве это трудовые споры, семейные дела, жилищные вопросы, имущественные претензии. На третьем, наиболее детальном уровне классификация производится по конкретному правовому основанию либо статье нормативного акта.

Качество классификации существенно зависит от двух факторов: репрезентативности обучающей выборки (наличие примеров каждой категории, сбалансированность классов) и качества текстового представления (feature engineering или end-to-end обучение признаков).

#### 2.2. Общий конвейер обработки

Автоматическая классификация судебных документов реализуется в виде последовательного конвейера (pipeline), включающего следующие этапы: сбор и подготовка данных, предобработка текстов, извлечение признаков, обучение модели и оценка качества. Каждый из этих этапов имеет принципиальное значение для итогового результата, и пренебрежение любым из них неизбежно приводит к деградации качества системы в целом.

Принципиальная схема конвейера может быть формализована следующим образом. Пусть $D = \{d_1, d_2, \ldots, d_N\}$ — коллекция необработанных текстовых документов. Функция предобработки $g: D \to D'$ преобразует исходные тексты в нормализованную форму. Функция извлечения признаков $\phi: D' \to \mathbb{R}^d$ отображает каждый документ в числовой вектор. Классификатор $f_\theta: \mathbb{R}^d \to \mathcal{Y}$ сопоставляет вектору признаков метку класса. Итоговый результат: $\hat{y}_i = f_\theta(\phi(g(d_i)))$.

---


### 3. Сбор и подготовка данных

#### 3.1. Источники юридических данных

Качество любой системы машинного обучения принципиально определяется качеством обучающих данных, поэтому этап сбора и подготовки датасета заслуживает особого внимания. В сфере автоматизации обработки судебных документов ключевыми источниками текстовой информации являются официальные государственные ресурсы, специализированные правовые базы данных и открытые архивы судебных решений.

К числу основных открытых источников судебной практики в России относится портал «ГАС Правосудие» (sudrf.ru), агрегирующий решения судов общей юрисдикции, а также система «Картотека арбитражных дел» (kad.arbitr.ru), предоставляющая доступ к документам арбитражных судов. Коммерческие правовые базы данных — КонсультантПлюс, Гарант, РИАС «Право» — предлагают более широкий функционал поиска и фильтрации, однако требуют лицензионного соглашения.

При организации сбора данных необходимо учитывать ряд технических и правовых ограничений. Во-первых, автоматизированный парсинг сайтов должен соответствовать правилам, зафиксированным в файле robots.txt, а также общим нормам авторского права и законодательства о персональных данных. Во-вторых, судебные решения могут содержать персональную информацию, которая подлежит обезличиванию в соответствии с требованиями законодательства о защите персональных данных (152-ФЗ). В-третьих, HTML-разметка официальных ресурсов нередко отличается непоследовательностью, что усложняет извлечение чистого текстового содержимого.

#### 3.2. Программный сбор данных (web scraping)

Для автоматизированного извлечения текстовых документов с веб-ресурсов применяется технология web scraping, реализуемая, как правило, с использованием библиотек `requests` и `BeautifulSoup` в экосистеме Python. Следует подчеркнуть, что перед разработкой скрапера необходимо изучить документацию соответствующего ресурса и убедиться в допустимости автоматизированного сбора данных.


In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import re
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def fetch_document(url: str, delay: float = 1.0) -> str:
    """
    Загружает и извлекает текстовое содержимое судебного документа
    по заданному URL. Параметр delay (секунды) задаёт паузу между
    запросами для соблюдения политики допустимой нагрузки на сервер.

    Args:
        url: Адрес страницы с документом.
        delay: Задержка между запросами (в секундах).

    Returns:
        Очищенный текст документа или пустая строка в случае ошибки.
    """
    try:
        headers = {
            'User-Agent': (
                'Mozilla/5.0 (compatible; ResearchBot/1.0; '
                '+https://university.example.ru/research)'
            )
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        response.encoding = 'utf-8'

        soup = BeautifulSoup(response.content, 'html.parser')

        # Удаляем нерелевантные элементы страницы
        for tag in soup.find_all(['script', 'style', 'nav', 'footer', 'header']):
            tag.decompose()

        # Извлекаем основной текст
        text = soup.get_text(separator=' ', strip=True)

        # Нормализация пробельных символов
        text = re.sub(r'\s+', ' ', text).strip()
        time.sleep(delay)
        return text

    except requests.RequestException as e:
        logger.error(f"Ошибка при загрузке {url}: {e}")
        return ""


def download_corpus(url_list: list, delay: float = 1.5) -> list:
    """
    Загружает коллекцию документов по списку URL-адресов.

    Args:
        url_list: Список URL-адресов страниц с документами.
        delay: Задержка между запросами.

    Returns:
        Список строк — текстовых содержимых документов.
    """
    corpus = []
    for idx, url in enumerate(url_list):
        logger.info(f"Загрузка документа {idx + 1}/{len(url_list)}: {url}")
        doc = fetch_document(url, delay=delay)
        if doc:
            corpus.append(doc)
    logger.info(f"Успешно загружено {len(corpus)} документов из {len(url_list)}.")
    return corpus


# Демонстрационный пример (URL являются условными)
sample_urls = [
    'https://example-court.ru/decisions/civil/doc001',
    'https://example-court.ru/decisions/criminal/doc002',
]
# corpus = download_corpus(sample_urls)
print("Функции сбора данных определены успешно.")
print(f"Пример запроса документа: fetch_document(url, delay=1.0)")



#### 3.3. Формирование размеченного датасета

После сбора сырых текстовых данных необходимо сформировать размеченный датасет для обучения с учителем. Разметка предполагает присвоение каждому документу одного или нескольких классовых меток. Разметку могут выполнять юристы-эксперты (ручная разметка), полуавтоматические системы (разметка на основе метаданных, уже присутствующих в структуре документа) или комбинированные подходы.

Важнейший аспект формирования датасета — обеспечение сбалансированности классов. Несбалансированные классы (class imbalance) являются типичной проблемой в юридических корпусах: количество административных дел может в десятки раз превышать число уголовных. Несбалансированность приводит к тому, что модель «научается» предсказывать мажоритарный класс, показывая высокую формальную точность (accuracy) при низком recall для миноритарных классов. Для борьбы с этой проблемой применяются методы over-sampling (SMOTE), under-sampling или взвешивание классов при обучении.

---


### 4. Предобработка текстов

#### 4.1. Теоретические основы предобработки

Предобработка (preprocessing) текстовых данных — обязательный этап любого конвейера анализа естественного языка. Целью предобработки является приведение сырых текстов к нормализованной, лингвистически последовательной форме, устранение шума (HTML-артефактов, опечаток, нерелевантных символов) и сокращение размерности признакового пространства за счёт слияния лексических вариантов одной смысловой единицы.

В контексте юридических текстов предобработка имеет свою специфику. Юридические термины, аббревиатуры, ссылки на нормативные акты (например, «ст. 119 УК РФ», «п. 1 ст. 130 ГК РФ») являются высокоинформативными признаками и не должны удаляться безосновательно — в отличие от общего NLP-конвейера, где цифры и специальные символы нередко исключаются полностью. Напротив, стандартные вводные конструкции («Именем Российской Федерации», «Суд, рассмотрев материалы дела») являются шаблонными и несут минимальную классификационную информацию.

#### 4.2. Этапы предобработки

Стандартный конвейер предобработки юридических текстов включает следующие последовательные операции.

**Очистка текста.** На данном этапе из сырого текста удаляются HTML-теги, управляющие символы, избыточные пробельные символы, а также артефакты форматирования (разрывы строк, колонтитулы, нумерация страниц).

**Токенизация.** Разбиение очищенного текста на минимальные смысловые единицы — токены (слова, знаки препинания, числа). Для русскоязычных текстов применяются специализированные токенизаторы, учитывающие особенности кириллического письма и русскоязычной пунктуации.

**Удаление стоп-слов.** Высокочастотные служебные слова (предлоги, союзы, частицы: «в», «на», «и», «но», «что»), не несущие классификационной информации, удаляются из текста для снижения размерности.

**Лемматизация или стемминг.** Приведение словоформ к базовой форме (лемме) обеспечивает объединение морфологических вариантов одного слова («судебное», «судебных», «судебном» → «судебный»). Для русского языка лемматизация предпочтительнее стемминга ввиду высокой флективности языка.


In [ ]:
import re
import nltk
import pymorphy2
from typing import List, Optional

# Скачиваем необходимые ресурсы NLTK (однократно)
# nltk.download('punkt')
# nltk.download('stopwords')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Инициализация морфологического анализатора для русского языка
morph = pymorphy2.MorphAnalyzer()

# Стоп-слова для русского языка
RUSSIAN_STOPWORDS = set([
    'в', 'на', 'и', 'с', 'по', 'к', 'за', 'у', 'о', 'из',
    'не', 'он', 'она', 'они', 'что', 'как', 'это', 'при',
    'от', 'до', 'а', 'но', 'или', 'же', 'ли', 'бы', 'что',
    'для', 'об', 'со', 'также', 'то', 'во', 'да', 'нет',
    'быть', 'суд', 'дело', 'рф', 'российский', 'федерация',
])


def clean_text(text: str) -> str:
    """
    Очищает текст от HTML-разметки, специальных символов
    и нормализует пробельные символы.

    Математическая модель: преобразование g_clean: T -> T',
    где T — пространство сырых строк, T' — пространство
    нормализованных строк.

    Args:
        text: Исходная строка текста.

    Returns:
        Очищенная строка.
    """
    # Удаление HTML-тегов
    text = re.sub(r'<[^>]+>', ' ', text)
    # Нормализация кавычек и дефисов
    text = re.sub(r'[«»„""]', '"', text)
    text = re.sub(r'[–—]', '-', text)
    # Удаление URL
    text = re.sub(r'https?://\S+', ' ', text)
    # Оставляем только буквы, цифры, пунктуацию
    text = re.sub(r'[^\w\s.,;:()/\-]', ' ', text)
    # Нормализация пробелов
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()


def lemmatize_text(
    text: str,
    remove_stopwords: bool = True,
    min_token_length: int = 3
) -> List[str]:
    """
    Выполняет токенизацию и лемматизацию русскоязычного текста
    с использованием морфологического анализатора pymorphy2.

    Формальная запись: phi_lemma(t) = [lemma(w) for w in tokens(t)
                                        if w not in stopwords
                                        and len(w) >= min_token_length]

    Args:
        text: Предварительно очищенный текст.
        remove_stopwords: Удалять ли стоп-слова.
        min_token_length: Минимальная длина токена (символов).

    Returns:
        Список лемм (нормализованных форм слов).
    """
    tokens = word_tokenize(text, language='russian')
    lemmas = []

    for token in tokens:
        # Оставляем только буквенные токены достаточной длины
        if not token.isalpha() or len(token) < min_token_length:
            continue
        # Лемматизация через pymorphy2
        parsed = morph.parse(token)
        if not parsed:
            continue
        lemma = parsed[0].normal_form

        if remove_stopwords and lemma in RUSSIAN_STOPWORDS:
            continue
        lemmas.append(lemma)

    return lemmas


def preprocess_document(text: str) -> str:
    """
    Полный конвейер предобработки: очистка -> лемматизация -> объединение.

    Args:
        text: Сырой текст документа.

    Returns:
        Нормализованная строка для последующего векторного представления.
    """
    cleaned = clean_text(text)
    lemmas = lemmatize_text(cleaned)
    return ' '.join(lemmas)


# Демонстрация конвейера предобработки
sample_text = """
    Именем Российской Федерации! Судья Ленинского районного суда,
    рассмотрев в открытом судебном заседании гражданское дело
    №2-1234/2023 по иску Иванова И.И. к ООО «Строймаркет»
    о взыскании задолженности по договору подряда...
"""

cleaned = clean_text(sample_text)
lemmas = lemmatize_text(cleaned)
result = preprocess_document(sample_text)

print("Исходный текст:")
print(sample_text.strip())
print("\nПосле очистки:")
print(cleaned[:200])
print("\nЛеммы (первые 20):")
print(lemmas[:20])
print("\nИтоговая строка для векторизации:")
print(result[:200])



#### 4.3. Оценка влияния предобработки на качество классификации

Важно понимать, что агрессивная предобработка не всегда улучшает качество модели. Удаление «лишних» слов может уничтожить контекстуально важную информацию. Поэтому в исследовательской практике принято проводить ablation study — серию экспериментов, в которых поочерёдно отключаются отдельные компоненты конвейера предобработки, а результирующее качество классификации измеряется на фиксированной тестовой выборке.

---


### 5. Векторное представление текстов (Feature Extraction)

#### 5.1. Проблема представления текста для алгоритмов машинного обучения

Алгоритмы машинного обучения оперируют числовыми векторами, тогда как исходные данные — судебные документы — представлены в виде последовательностей символов. Центральная задача этапа извлечения признаков заключается в построении отображения $\phi: D \to \mathbb{R}^d$, переводящего текстовый документ в числовой вектор фиксированной размерности $d$. Качество этого отображения непосредственно определяет верхнюю границу достижимого качества классификации: никакой, сколь угодно мощный, классификатор не компенсирует потерю информации, допущенную на этапе векторизации.

#### 5.2. Модель «мешок слов» (Bag of Words)

Простейшим способом представления документа является модель «мешок слов» (Bag of Words, BoW), в которой документ представляется вектором частот вхождения каждого слова из словаря $V$. Пусть $|V| = m$ — размер словаря. Тогда документ $d_i$ представляется вектором:

$$\phi_{\text{BoW}}(d_i) = (\text{tf}(t_1, d_i), \text{tf}(t_2, d_i), \ldots, \text{tf}(t_m, d_i)) \in \mathbb{R}^m$$

где $\text{tf}(t, d)$ — частота (term frequency) термина $t$ в документе $d$:

$$\text{tf}(t, d) = \frac{\text{count}(t, d)}{\sum_{t' \in d} \text{count}(t', d)}$$

Несмотря на концептуальную простоту, BoW теряет информацию о порядке слов и синтаксической структуре предложений, что существенно ограничивает его возможности для задач, требующих понимания смысла.

#### 5.3. TF-IDF (Term Frequency–Inverse Document Frequency)

Метрика TF-IDF является усовершенствованием модели BoW, учитывающим специфичность термина для конкретного документа на фоне всей коллекции. Интуиция метода: слово, часто встречающееся в данном документе, но редкое во всей коллекции, несёт больше информации о специфике документа, чем слово, одинаково распространённое во всех документах.

TF-IDF вес термина $t$ в документе $d$ вычисляется как:

$$\text{tfidf}(t, d, D) = \text{tf}(t, d) \cdot \text{idf}(t, D)$$

где обратная документная частота (inverse document frequency) определяется следующим образом:

$$\text{idf}(t, D) = \log \frac{|D|}{1 + |\{d \in D : t \in d\}|}$$

Слагаемое $1$ в знаменателе предотвращает деление на нуль для термина, не встречающегося ни в одном документе. Логарифмирование демпфирует влияние экстремальных значений.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Демонстрационный корпус судебных документов (упрощённый)
corpus = [
    "суд отказал иск взыскание задолженность договор подряд",
    "суд удовлетворил иск признание недействительным сделка купля продажа",
    "суд прекратил производство дело нарушение порядок подача",
    "обвиняемый осуждён кража проникновение жилище наказание лишение свободы",
    "суд оправдал обвиняемый недоказанность вина отсутствие состав преступление",
    "административный штраф нарушение дорожное движение превышение скорость",
    "арбитражный суд удовлетворил требование банкротство ликвидация должник",
    "суд назначил экспертиза установление стоимость имущество спор",
]

labels = [
    'гражданское', 'гражданское', 'гражданское',
    'уголовное', 'уголовное',
    'административное',
    'арбитражное', 'арбитражное',
]

# ──────────────────────────────────────────────────────────────
# 1. Модель «мешок слов» (Bag of Words)
# ──────────────────────────────────────────────────────────────
bow_vectorizer = CountVectorizer(max_features=20, min_df=1)
X_bow = bow_vectorizer.fit_transform(corpus)

print("=== Модель «Мешок слов» (BoW) ===")
print(f"Размер матрицы документ-термин: {X_bow.shape}")
print(f"Размер словаря: {len(bow_vectorizer.vocabulary_)}")
print("Первые 10 терминов в словаре:")
vocab_sorted = sorted(bow_vectorizer.vocabulary_.items(), key=lambda x: x[1])
for term, idx in vocab_sorted[:10]:
    print(f"  [{idx:2d}] '{term}'")

# ──────────────────────────────────────────────────────────────
# 2. TF-IDF представление
# ──────────────────────────────────────────────────────────────
tfidf_vectorizer = TfidfVectorizer(
    max_features=20,
    min_df=1,
    sublinear_tf=True   # Применяет логарифмическое сглаживание TF
)
X_tfidf = tfidf_vectorizer.fit_transform(corpus)

print("\n=== TF-IDF представление ===")
print(f"Размер матрицы признаков: {X_tfidf.shape}")

# Наиболее информативные термины для каждого документа
feature_names = tfidf_vectorizer.get_feature_names_out()
print("\nТоп-3 информативных термина для каждого документа:")
for i, (doc, label) in enumerate(zip(corpus, labels)):
    row = X_tfidf[i].toarray().flatten()
    top_idx = np.argsort(row)[::-1][:3]
    top_terms = [(feature_names[j], round(row[j], 4)) for j in top_idx if row[j] > 0]
    print(f"  [{label}]: {top_terms}")



#### 5.4. Word Embeddings: дистрибутивные семантические модели

Принципиальным ограничением методов BoW и TF-IDF является пренебрежение семантической близостью слов: в пространстве TF-IDF векторы «суд» и «трибунал», «иск» и «претензия» ортогональны, хотя семантически они близки. Дистрибутивные семантические модели (word embeddings) устраняют этот недостаток, обучая плотное низкоразмерное представление каждого слова, в котором геометрическая близость векторов отражает семантическую близость слов.

Модель Word2Vec (Mikolov et al., 2013) обучает вектор $\mathbf{v}_w \in \mathbb{R}^{d_e}$ для каждого слова $w \in V$ путём максимизации логарифмического правдоподобия контекстного предсказания. В версии Skip-gram модель предсказывает контекстные слова по центральному:

$$\mathcal{L}_{\text{SG}} = \sum_{t=1}^{T} \sum_{-c \leq j \leq c,\, j \neq 0} \log p(w_{t+j} | w_t)$$

где вероятность контекстного слова вычисляется через softmax:

$$p(w_O | w_I) = \frac{\exp(\mathbf{v}_{w_O}^\top \mathbf{v}_{w_I}')}{\sum_{w=1}^{|V|} \exp(\mathbf{v}_w^\top \mathbf{v}_{w_I}')}$$

Документный вектор в такой модели может быть получен как среднее по всем словам документа: $\phi_{\text{W2V}}(d) = \frac{1}{|d|} \sum_{w \in d} \mathbf{v}_w$.

---


### 6. Методы классификации

#### 6.1. Логистическая регрессия

Логистическая регрессия является одним из фундаментальных методов классификации, широко применяемых в задачах обработки текстов благодаря интерпретируемости, вычислительной эффективности и хорошей устойчивости к переобучению при наличии большого числа признаков (что типично для TF-IDF представлений).

В задаче многоклассовой классификации с $K$ категориями модель вычисляет вероятность принадлежности к каждому классу через функцию softmax:

$$P(y = k \mid x; W, b) = \frac{\exp(w_k^\top x + b_k)}{\sum_{j=1}^{K} \exp(w_j^\top x + b_j)}, \quad k = 1, \ldots, K$$

где $W = [w_1, \ldots, w_K]^\top \in \mathbb{R}^{K \times d}$ — матрица весов, $b \in \mathbb{R}^K$ — вектор смещений. Параметры оцениваются минимизацией кросс-энтропийных потерь с $L_2$-регуляризацией:

$$\mathcal{L}(W, b) = -\frac{1}{N} \sum_{i=1}^{N} \sum_{k=1}^{K} \mathbb{1}[y_i = k] \log P(y = k \mid x_i) + \lambda \|W\|_F^2$$

где $\lambda > 0$ — гиперпараметр регуляризации, $\|W\|_F$ — норма Фробениуса.

#### 6.2. Метод опорных векторов (SVM)

Метод опорных векторов (Support Vector Machine, SVM) ищет гиперплоскость $\{x: w^\top x + b = 0\}$, максимизирующую зазор (margin) $\frac{2}{\|w\|}$ между классами. Для нелинейно разделимых данных применяется «мягкий» вариант с переменными отступа $\xi_i \geq 0$:

$$\min_{w, b, \xi} \frac{1}{2}\|w\|^2 + C \sum_{i=1}^{N} \xi_i$$
$$\text{s.t.} \quad y_i(w^\top x_i + b) \geq 1 - \xi_i, \quad \xi_i \geq 0$$

Параметр $C > 0$ управляет компромиссом между максимизацией зазора и штрафом за ошибки классификации. SVM с линейным ядром исторически демонстрировал лучшие результаты на задачах классификации текстов при BoW/TF-IDF представлении, что объясняется высокой размерностью и разреженностью признакового пространства.

#### 6.3. Случайный лес (Random Forest)

Случайный лес — ансамблевый метод, строящий множество деревьев решений $\{T_1, T_2, \ldots, T_B\}$ на бутстрап-выборках из обучающего набора и использующий мажоритарное голосование для финального предсказания:

$$\hat{y} = \text{mode}\{T_1(x), T_2(x), \ldots, T_B(x)\}$$

Декорреляция деревьев достигается случайным выбором подмножества из $m = \lfloor\sqrt{d}\rfloor$ признаков при каждом разбиении узла. Случайные леса устойчивы к выбросам, не требуют нормализации признаков и предоставляют оценки важности признаков (feature importances), полезные для интерпретации модели.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
import warnings
warnings.filterwarnings('ignore')

# ──────────────────────────────────────────────────────────────
# Демонстрационный датасет судебных документов
# ──────────────────────────────────────────────────────────────
import random
random.seed(42)
np.random.seed(42)

# Шаблоны документов для генерации учебного корпуса
templates = {
    'гражданское': [
        "суд решил взыскать задолженность по договору с ответчика в пользу истца",
        "иск о признании сделки недействительной удовлетворить частично",
        "обязать ответчика возместить причинённый ущерб имуществу истца",
        "в удовлетворении требований о расторжении договора аренды отказать",
        "взыскать алименты на содержание несовершеннолетнего ребёнка",
    ],
    'уголовное': [
        "осудить за совершение кражи с незаконным проникновением в жилище",
        "приговорить к лишению свободы условно с испытательным сроком",
        "оправдать в связи с отсутствием состава преступления в действиях",
        "переквалифицировать с грабежа на хулиганство с изменением наказания",
        "назначить наказание в виде обязательных работ за административное правонарушение",
    ],
    'административное': [
        "назначить административный штраф за нарушение правил дорожного движения",
        "предупреждение за нарушение санитарных норм без назначения штрафа",
        "прекратить производство по делу об административном правонарушении",
        "наложить административный арест сроком на пятнадцать суток",
        "штраф за несоблюдение требований пожарной безопасности в организации",
    ],
    'арбитражное': [
        "признать должника несостоятельным и открыть конкурсное производство",
        "взыскать неустойку по договору поставки продукции между организациями",
        "отказать в иске о признании договора подряда незаключённым",
        "утвердить мировое соглашение между сторонами корпоративного спора",
        "обязать налоговый орган возвратить излишне уплаченный налог",
    ],
}

# Генерация корпуса из 200 документов
texts, labels = [], []
for label, tmpl_list in templates.items():
    for _ in range(50):  # 50 документов каждого класса
        base = random.choice(tmpl_list)
        # Добавляем незначительное разнообразие
        extra_words = random.choice([
            "рассмотрев материалы дела вынес решение",
            "исследовав доказательства установил обстоятельства",
            "выслушав стороны изучив документы постановил",
        ])
        texts.append(f"{base} {extra_words}")
        labels.append(label)

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels
)

print(f"Обучающая выборка: {len(X_train)} документов")
print(f"Тестовая выборка:  {len(X_test)} документов")
print(f"Классы: {sorted(set(labels))}\n")

# ──────────────────────────────────────────────────────────────
# Конвейеры классификации
# ──────────────────────────────────────────────────────────────
classifiers = {
    'Логистическая регрессия': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000,
                                  sublinear_tf=True)),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=42)),
    ]),
    'Линейный SVM': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000,
                                  sublinear_tf=True)),
        ('clf', LinearSVC(C=1.0, max_iter=2000, random_state=42)),
    ]),
    'Случайный лес': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 1), max_features=3000)),
        ('clf', RandomForestClassifier(
            n_estimators=100, max_depth=20, random_state=42, n_jobs=-1
        )),
    ]),
}

results = {}
for name, pipeline in classifiers.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{'=' * 55}")
    print(f"Модель: {name}")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, zero_division=0))

print("\n=== Сравнение моделей ===")
for name, acc in sorted(results.items(), key=lambda x: -x[1]):
    bar = '█' * int(acc * 40)
    print(f"  {name:<30} {acc:.4f}  {bar}")



#### 6.4. Анализ и интерпретация результатов

При сравнении классических методов классификации на задаче категоризации судебных документов следует обращать внимание не только на агрегированную метрику Accuracy, но и на значения Precision и Recall для каждого класса в отдельности. На практике цена ошибки существенно варьируется в зависимости от типа ошибочной классификации: ложная классификация уголовного дела как административного несёт принципиально иные практические последствия, чем смешение гражданских и арбитражных категорий.

---


### 7. Нейросетевые методы классификации судебных документов

#### 7.1. Рекуррентные нейронные сети (RNN)

В отличие от методов BoW и TF-IDF, нейросетевые подходы способны учитывать последовательный характер текста — порядок слов и долгосрочные зависимости между ними. Рекуррентные нейронные сети (Recurrent Neural Networks, RNN) представляют собой архитектуру, специально предназначенную для обработки последовательностей данных посредством поддержания скрытого состояния $h_t$, передающего информацию от предыдущих шагов к последующим.

На каждом временном шаге $t$ стандартный рекуррентный слой вычисляет скрытое состояние $h_t \in \mathbb{R}^{d_h}$ по формуле:

$$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$$

где $x_t \in \mathbb{R}^{d_e}$ — эмбеддинг $t$-го слова, $W_{hh} \in \mathbb{R}^{d_h \times d_h}$ — матрица рекуррентных весов, $W_{xh} \in \mathbb{R}^{d_h \times d_e}$ — матрица входных весов. Классификационный ответ формируется на основе последнего скрытого состояния $h_T$:

$$P(y = k \mid x_{1:T}) = \text{softmax}(W_o h_T + b_o)$$

Фундаментальной проблемой классических RNN является явление исчезающего (и взрывного) градиента при обучении методом BPTT (Backpropagation Through Time). Градиент ошибки при распространении на $T$ шагов назад масштабируется как произведение матриц $\prod_{t=1}^{T} W_{hh}$, что при $T \gg 1$ ведёт к экспоненциальному затуханию или взрыву сигнала.

#### 7.2. Сети с долгой краткосрочной памятью (LSTM)

Архитектура LSTM (Long Short-Term Memory, Hochreiter & Schmidhuber, 1997) решает проблему исчезающего градиента посредством явного механизма управляемой памяти. Ключевым элементом LSTM является ячейка состояния $c_t \in \mathbb{R}^{d_h}$, информация в которой сохраняется через длинные последовательности с минимальными модификациями.

Обновление состояний LSTM на шаге $t$ определяется системой уравнений:

$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i) \quad \text{(вентиль ввода)}$$
$$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f) \quad \text{(вентиль забывания)}$$
$$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o) \quad \text{(вентиль вывода)}$$
$$\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c) \quad \text{(кандидат состояния)}$$
$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \quad \text{(обновление ячейки)}$$
$$h_t = o_t \odot \tanh(c_t) \quad \text{(скрытое состояние)}$$

где $\sigma$ — сигмоидная функция активации, $\odot$ — поэлементное произведение (Адамар). Вентиль забывания $f_t \in (0,1)^{d_h}$ определяет, какая доля предыдущего состояния ячейки $c_{t-1}$ сохраняется.


In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ──────────────────────────────────────────────────────────────
# Реализация LSTM-классификатора для текстов на Keras/TensorFlow
# ──────────────────────────────────────────────────────────────

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import (
        Embedding, LSTM, Dense, Dropout, Bidirectional, GlobalMaxPooling1D
    )
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.utils import to_categorical
    from sklearn.preprocessing import LabelEncoder
    TF_AVAILABLE = True
except ImportError:
    TF_AVAILABLE = False
    print("TensorFlow не установлен. Показываем архитектуру в псевдокоде.")

if TF_AVAILABLE:
    # ── Параметры модели ──
    VOCAB_SIZE   = 3000    # Размер словаря
    EMBED_DIM    = 128     # Размерность эмбеддинга
    MAX_LEN      = 30      # Максимальная длина последовательности
    LSTM_UNITS   = 64      # Число нейронов LSTM
    NUM_CLASSES  = 4       # Число категорий
    DROPOUT_RATE = 0.3
    EPOCHS       = 10
    BATCH_SIZE   = 32

    # ── Подготовка данных ──
    label_encoder = LabelEncoder()
    y_enc = label_encoder.fit_transform(labels)
    y_cat = to_categorical(y_enc, num_classes=NUM_CLASSES)

    tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<UNK>')
    tokenizer.fit_on_texts(texts)
    sequences = tokenizer.texts_to_sequences(texts)
    X_padded  = pad_sequences(sequences, maxlen=MAX_LEN, padding='post',
                               truncating='post')

    split = int(0.75 * len(X_padded))
    X_tr, X_te = X_padded[:split], X_padded[split:]
    y_tr, y_te = y_cat[:split], y_cat[split:]

    # ── Построение двунаправленного LSTM ──
    model = Sequential([
        Embedding(
            input_dim=VOCAB_SIZE,
            output_dim=EMBED_DIM,
            input_length=MAX_LEN,
            name='embedding'
        ),
        # Bidirectional LSTM захватывает контекст в обоих направлениях:
        # h_t = [h_t^{forward}; h_t^{backward}]
        Bidirectional(LSTM(LSTM_UNITS, return_sequences=True), name='biLSTM'),
        GlobalMaxPooling1D(name='maxpool'),
        Dropout(DROPOUT_RATE, name='dropout'),
        Dense(32, activation='relu', name='dense_hidden'),
        Dense(NUM_CLASSES, activation='softmax', name='output'),
    ], name='BiLSTM_Classifier')

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    model.summary()

    # ── Обучение ──
    history = model.fit(
        X_tr, y_tr,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_te, y_te),
        verbose=1,
    )

    # ── Оценка качества ──
    loss, acc = model.evaluate(X_te, y_te, verbose=0)
    print(f"\nBiLSTM — Тестовая точность: {acc:.4f}")
    print(f"BiLSTM — Тестовые потери:    {loss:.4f}")

else:
    # Псевдокод архитектуры
    print("""
=== Архитектура двунаправленного LSTM (псевдокод) ===

model = Sequential([
    Embedding(VOCAB_SIZE=3000, EMBED_DIM=128, MAX_LEN=30),
    Bidirectional(LSTM(units=64, return_sequences=True)),
    GlobalMaxPooling1D(),
    Dropout(rate=0.3),
    Dense(32, activation='relu'),
    Dense(NUM_CLASSES=4, activation='softmax'),
])

# Функция потерь: H(y, p) = -sum_k y_k * log(p_k)
# Оптимизатор: Adam (lr=0.001, beta_1=0.9, beta_2=0.999)
""")



#### 7.3. Трансформерные архитектуры и BERT

Современным стандартом в задачах обработки естественного языка являются трансформерные модели, основанные на механизме самовнимания (self-attention). Архитектура BERT (Bidirectional Encoder Representations from Transformers, Devlin et al., 2019) и её русскоязычные адаптации (RuBERT, ruBERT-tiny) обеспечивают существенное превосходство над LSTM-подходами за счёт предобучения на сверхбольших корпусах.

Механизм многоголового внимания (Multi-Head Attention) вычисляет взвешенное агрегирование значений $V$ с весами, определяемыми сходством запросов $Q$ и ключей $K$:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

где нормировка на $\sqrt{d_k}$ предотвращает насыщение функции softmax при больших значениях скалярных произведений. Многоголовое внимание объединяет $H$ параллельных механизмов:

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_H) W^O$$

Для тонкой настройки (fine-tuning) предобученного BERT на задаче классификации судебных документов достаточно добавить один классификационный слой поверх токена [CLS] и обучить всю систему на размеченной юридической выборке.

---


### 8. Оценка качества модели классификации

#### 8.1. Матрица ошибок (Confusion Matrix)

Матрица ошибок (confusion matrix) $C \in \mathbb{R}^{K \times K}$ является наиболее полным инструментом анализа качества многоклассового классификатора. Элемент $C_{ij}$ показывает число объектов истинного класса $i$, отнесённых моделью к классу $j$. Диагональные элементы $C_{ii}$ соответствуют корректным классификациям, внедиагональные — ошибкам.

#### 8.2. Метрики качества

Из матрицы ошибок извлекаются стандартные метрики качества для каждого класса $k$:

Точность (Precision) отражает долю истинно положительных среди всех случаев, отнесённых к классу $k$:

$$\text{Precision}_k = \frac{C_{kk}}{\sum_{i=1}^{K} C_{ik}}$$

Полнота (Recall) отражает долю истинно положительных среди всех объектов класса $k$:

$$\text{Recall}_k = \frac{C_{kk}}{\sum_{j=1}^{K} C_{kj}}$$

Мера $F_1$ представляет собой гармоническое среднее Precision и Recall:

$$F_1^{(k)} = \frac{2 \cdot \text{Precision}_k \cdot \text{Recall}_k}{\text{Precision}_k + \text{Recall}_k}$$

Обобщённая $F_\beta$-мера позволяет управлять балансом между Precision и Recall через параметр $\beta$:

$$F_\beta^{(k)} = (1 + \beta^2) \cdot \frac{\text{Precision}_k \cdot \text{Recall}_k}{\beta^2 \cdot \text{Precision}_k + \text{Recall}_k}$$

При $\beta > 1$ усиливается вклад Recall (критично при высокой цене пропуска), при $\beta < 1$ — Precision (критично при высокой цене ложных тревог).

#### 8.3. Кросс-валидация

Для получения надёжных оценок качества применяется $k$-кратная кросс-валидация ($k$-fold cross-validation). Обучающий набор разбивается на $k$ равных частей; модель обучается на $k-1$ частях и тестируется на оставшейся, операция повторяется $k$ раз. Итоговая оценка качества и её дисперсия вычисляются по $k$ независимым измерениям:

$$\hat{\mathcal{L}} = \frac{1}{k} \sum_{i=1}^{k} \mathcal{L}_i, \quad \hat{\sigma}^2 = \frac{1}{k-1} \sum_{i=1}^{k} (\mathcal{L}_i - \hat{\mathcal{L}})^2$$

Значение $k = 5$ или $k = 10$ является стандартным выбором в исследовательской практике.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# ── Обучение финальной модели ──
best_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000,
                               sublinear_tf=True)),
    ('clf',   LogisticRegression(C=1.0, max_iter=1000, random_state=42)),
])
best_pipeline.fit(X_train, y_train)
y_pred = best_pipeline.predict(X_test)

class_names = sorted(set(labels))
print("=== Подробный отчёт о качестве классификации ===")
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))

# ── Матрица ошибок ──
cm = confusion_matrix(y_test, y_pred, labels=class_names)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Тепловая карта матрицы ошибок
ax = axes[0]
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set(
    xticks=range(len(class_names)),
    yticks=range(len(class_names)),
    xticklabels=class_names,
    yticklabels=class_names,
    xlabel='Предсказанный класс',
    ylabel='Истинный класс',
    title='Матрица ошибок (Confusion Matrix)',
)
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
thresh = cm.max() / 2.0
for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(j, i, f'{cm[i,j]}',
                ha='center', va='center',
                color='white' if cm[i,j] > thresh else 'black',
                fontsize=12, fontweight='bold')

# ── Кросс-валидация ──
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_pipeline, texts, labels, cv=cv,
                             scoring='f1_weighted', n_jobs=-1)

ax2 = axes[1]
folds = np.arange(1, len(cv_scores) + 1)
colors = ['#2196F3' if s >= cv_scores.mean() else '#90CAF9' for s in cv_scores]
bars = ax2.bar(folds, cv_scores, color=colors, edgecolor='navy', linewidth=0.8)
ax2.axhline(cv_scores.mean(), color='red', linestyle='--', linewidth=1.5,
             label=f'Среднее F₁ = {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
ax2.fill_between(
    [0.4, len(cv_scores) + 0.6],
    cv_scores.mean() - cv_scores.std(),
    cv_scores.mean() + cv_scores.std(),
    color='red', alpha=0.12, label='±σ'
)
ax2.set(
    xlabel='Фолд',
    ylabel='F₁-weighted',
    title='5-кратная стратифицированная кросс-валидация',
    xticks=folds,
    ylim=[max(0, cv_scores.min() - 0.05), min(1, cv_scores.max() + 0.05)],
)
ax2.legend(fontsize=9)
for bar, score in zip(bars, cv_scores):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
              f'{score:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/evaluation_plots.png', dpi=150,
            bbox_inches='tight')
plt.show()
print("\n=== Кросс-валидация (5-fold, F₁-weighted) ===")
print(f"Оценки по фолдам: {[round(s, 4) for s in cv_scores]}")
print(f"Среднее:          {cv_scores.mean():.4f}")
print(f"Стандартное откл: {cv_scores.std():.4f}")


---

### 9. Обнаружение аномалий в судебных документах

#### 9.1. Постановка задачи и практическое значение

Обнаружение аномалий (anomaly detection, novelty detection) в судебных документах представляет собой самостоятельный класс задач, принципиально отличающийся от классификации. Если классификация предполагает наличие размеченных обучающих примеров каждой категории, то обнаружение аномалий решает задачу идентификации объектов, чьи свойства существенно отклоняются от типичного распределения, при минимальном числе размеченных аномалий или их полном отсутствии на этапе обучения.

Практическое значение обнаружения аномалий в правовой сфере является весьма широким. Анализ судебных решений может выявлять нетипичные формулировки, связанные с нарушениями процессуальных норм или несоответствием установившейся судебной практике. Обнаружение аномальных паттернов в временных рядах судебных решений позволяет идентифицировать возможную коррупционную деятельность или системные сбои в правоприменении. В сфере правового комплаенса идентификация нестандартных договорных условий помогает минимизировать юридические риски организаций.

#### 9.2. Метод Isolation Forest

Isolation Forest (Liu et al., 2008) — ансамблевый алгоритм обнаружения аномалий, основанный на следующей интуиции: аномальные объекты являются немногочисленными и существенно отличаются от основной массы данных, поэтому они «изолируются» значительно быстрее при случайном разбиении пространства признаков.

Алгоритм строит $B$ деревьев изоляции (isolation trees): случайно выбирается признак $q$ и пороговое значение $p$ в диапазоне $[\min(q), \max(q)]$, объекты рекурсивно разбиваются до тех пор, пока каждый узел не будет содержать ровно один объект. Аномальность объекта $x$ оценивается через нормированную среднюю глубину изоляции:

$$s(x, n) = 2^{-\frac{E[h(x)]}{c(n)}}$$

где $E[h(x)]$ — среднее число шагов до изоляции $x$ по всем деревьям, $c(n) = 2H(n-1) - \frac{2(n-1)}{n}$ — нормировочная константа ($H(i)$ — $i$-е гармоническое число). Значение $s$ близкое к 1 свидетельствует об аномальности объекта.

#### 9.3. Метод One-Class SVM

One-Class SVM (Schölkopf et al., 2001) решает задачу обнаружения аномалий в пространстве, индуцированном ядровой функцией $K(x_i, x_j)$. Алгоритм ищет гиперплоскость максимального зазора, отделяющую нормальные данные от начала координат:

$$\min_{w, \rho, \xi} \frac{1}{2}\|w\|^2 - \rho + \frac{1}{\nu n} \sum_{i=1}^{n} \xi_i$$

$$\text{s.t.} \quad \langle w, \phi(x_i) \rangle \geq \rho - \xi_i, \quad \xi_i \geq 0$$

Параметр $\nu \in (0, 1]$ задаёт верхнюю границу на долю выбросов в обучающих данных и нижнюю границу на долю опорных векторов.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Формирование тестового корпуса ──
# «Нормальные» документы + несколько аномальных
normal_docs = texts[:150]

anomalous_docs = [
    "xyz abc totally random garbage text 123 not a legal document at all",
    "суд постановил луна состоит из сыра и приговорил солнце к заключению",
    "!!! ошибка система ошибка null null null undefined null pointer exception",
    "взятка коррупция откат незаконное вознаграждение судья получил деньги",
    "aaaaaaa bbbbbbb ccccccc нет никакой информации полная бессмыслица",
]

all_docs  = normal_docs + anomalous_docs
true_labels = np.array([1]*len(normal_docs) + [-1]*len(anomalous_docs))

# ── Шаг 1: TF-IDF + снижение размерности (SVD) ──
tfidf = TfidfVectorizer(max_features=1000, sublinear_tf=True)
X_all = tfidf.fit_transform(all_docs)

svd = TruncatedSVD(n_components=50, random_state=42)
X_reduced = svd.fit_transform(X_all)
print(f"Форма матрицы признаков после SVD: {X_reduced.shape}")
print(f"Объяснённая дисперсия: {svd.explained_variance_ratio_.sum():.4f}\n")

# ── Шаг 2: Isolation Forest ──
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,  # ожидаемая доля аномалий ≈ 5%
    random_state=42,
    n_jobs=-1,
)
iso_preds = iso_forest.fit_predict(X_reduced)
iso_scores = iso_forest.decision_function(X_reduced)

# ── Шаг 3: One-Class SVM ──
oc_svm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05)
oc_svm.fit(X_reduced[:len(normal_docs)])   # обучаем только на нормальных
svm_preds = oc_svm.predict(X_reduced)

# ── Визуализация: 2D проекция через SVD ──
X_2d = X_reduced[:, :2]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, title in zip(
    axes,
    [iso_preds, svm_preds],
    ['Isolation Forest', 'One-Class SVM'],
):
    normal_mask = preds == 1
    anom_mask   = preds == -1
    ax.scatter(
        X_2d[normal_mask, 0], X_2d[normal_mask, 1],
        c='steelblue', s=30, alpha=0.6, label='Нормальные'
    )
    ax.scatter(
        X_2d[anom_mask, 0], X_2d[anom_mask, 1],
        c='crimson', s=80, marker='X', zorder=5,
        edgecolors='darkred', linewidths=0.7, label='Аномалии'
    )
    ax.set(title=title, xlabel='SVD-1', ylabel='SVD-2')
    ax.legend(fontsize=9)
    # Подписываем известные аномалии
    for idx in np.where(true_labels == -1)[0]:
        ax.annotate(
            f'A{idx - len(normal_docs) + 1}',
            X_2d[idx],
            textcoords='offset points',
            xytext=(5, 5), fontsize=8, color='darkred'
        )

plt.suptitle('Обнаружение аномалий в корпусе судебных документов',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/anomaly_detection.png', dpi=150,
            bbox_inches='tight')
plt.show()

# ── Отчёт об обнаруженных аномалиях ──
print("=== Результаты Isolation Forest ===")
print(f"Обнаружено аномалий: {(iso_preds == -1).sum()} из {len(all_docs)} документов")
print("\nАномальные документы:")
for idx in np.where(iso_preds == -1)[0]:
    print(f"  [{idx}] score={iso_scores[idx]:.4f}: {all_docs[idx][:80]}...")

print("\n=== Результаты One-Class SVM ===")
print(f"Обнаружено аномалий: {(svm_preds == -1).sum()} из {len(all_docs)} документов")


---

### 10. Визуализация и анализ результатов

#### 10.1. Роль визуализации в анализе текстовых данных

Визуализация является неотъемлемым инструментом исследования высокоразмерных данных, к которым относятся текстовые представления. Непосредственная интерпретация матрицы TF-IDF размерностью $N \times d$ (где $d$ может превышать десятки тысяч) невозможна без предварительного снижения размерности. Методы нелинейного снижения размерности позволяют проецировать высокоразмерные данные на двумерное пространство, сохраняя локальную структуру исходного распределения и обнажая кластерную организацию данных.

#### 10.2. Метод t-SNE

Алгоритм t-SNE (t-distributed Stochastic Neighbour Embedding, van der Maaten & Hinton, 2008) минимизирует дивергенцию Кульбака–Лейблера между распределениями попарных расстояний в исходном и целевом пространствах. В исходном пространстве сходство объектов $x_i$ и $x_j$ выражается через условную вероятность:

$$p_{j|i} = \frac{\exp(-\|x_i - x_j\|^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-\|x_i - x_k\|^2 / 2\sigma_i^2)}$$

В целевом пространстве применяется распределение Стьюдента с одной степенью свободы (тяжёлые хвосты предотвращают проблему скопления точек):

$$q_{ij} = \frac{(1 + \|y_i - y_j\|^2)^{-1}}{\sum_{k \neq l} (1 + \|y_k - y_l\|^2)^{-1}}$$

Минимизируемый функционал: $\mathcal{C} = KL(P \| Q) = \sum_{i,j} p_{ij} \log \frac{p_{ij}}{q_{ij}}$.

Гиперпараметр perplexity контролирует эффективное число ближайших соседей и обычно задаётся в диапазоне $[5, 50]$.

#### 10.3. Облако слов (Word Cloud)

Облако слов является популярным инструментом визуализации частотных характеристик текстового корпуса: размер слова пропорционален его TF-IDF весу или абсолютной частоте. Несмотря на ограниченные аналитические возможности, облака слов эффективны для первичного исследования тематического содержания корпуса и выявления ключевых терминов.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter

# ── Подготовка данных ──
tfidf_vis = TfidfVectorizer(max_features=500, sublinear_tf=True, min_df=1)
X_vis = tfidf_vis.fit_transform(texts)

# SVD -> 50 компонент для ускорения t-SNE
svd_vis = TruncatedSVD(n_components=50, random_state=42)
X_svd = svd_vis.fit_transform(X_vis)

# t-SNE: снижение до 2D
tsne = TSNE(
    n_components=2,
    perplexity=20,
    n_iter=1000,
    random_state=42,
    learning_rate='auto',
    init='pca',
)
X_tsne = tsne.fit_transform(X_svd)

# ── Палитра и маркеры ──
class_palette = {
    'гражданское':    ('#1976D2', 'o'),
    'уголовное':      ('#D32F2F', 's'),
    'административное': ('#388E3C', '^'),
    'арбитражное':    ('#F57C00', 'D'),
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── t-SNE визуализация ──
ax1 = axes[0]
for cls, (color, marker) in class_palette.items():
    idx = [i for i, l in enumerate(labels) if l == cls]
    ax1.scatter(
        X_tsne[idx, 0], X_tsne[idx, 1],
        c=color, marker=marker, s=45, alpha=0.75,
        edgecolors='white', linewidths=0.4, label=cls.capitalize()
    )
ax1.set(
    title='t-SNE визуализация TF-IDF представлений\nсудебных документов',
    xlabel='t-SNE компонента 1',
    ylabel='t-SNE компонента 2',
)
ax1.legend(fontsize=9, title='Категория дела', title_fontsize=9)
ax1.grid(alpha=0.3, linestyle='--')

# ── Топ-15 термины по категориям ──
ax2 = axes[1]
category_terms = {}
for cls in class_palette:
    cat_texts = [texts[i] for i, l in enumerate(labels) if l == cls]
    tv = TfidfVectorizer(max_features=15, sublinear_tf=True)
    tv.fit(cat_texts)
    # Средний TF-IDF вес термина по корпусу класса
    X_cls = tv.transform(cat_texts)
    mean_tfidf = np.asarray(X_cls.mean(axis=0)).flatten()
    feature_names = tv.get_feature_names_out()
    top_idx = np.argsort(mean_tfidf)[::-1][:8]
    category_terms[cls] = [(feature_names[j], mean_tfidf[j]) for j in top_idx]

# Горизонтальная диаграмма для одного класса (гражданского)
cls_show = 'уголовное'
terms_data = category_terms[cls_show]
terms_sorted = sorted(terms_data, key=lambda x: x[1])
names_ = [t[0] for t in terms_sorted]
values_ = [t[1] for t in terms_sorted]
color_, _ = class_palette[cls_show]

bars = ax2.barh(names_, values_, color=color_, alpha=0.85,
                edgecolor='white', linewidth=0.5)
ax2.set(
    title=f'Ключевые термины категории «{cls_show}»\n(средний TF-IDF вес)',
    xlabel='Средний TF-IDF',
    xlim=[0, max(values_) * 1.15],
)
for bar, val in zip(bars, values_):
    ax2.text(val + 0.001, bar.get_y() + bar.get_height()/2,
              f'{val:.4f}', va='center', fontsize=8)
ax2.grid(axis='x', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/visualization.png', dpi=150,
            bbox_inches='tight')
plt.show()
print("Визуализация сохранена в /mnt/user-data/outputs/visualization.png")


---

### 11. Комплексный практический пример

#### 11.1. Постановка задачи

В данном разделе представлен полный пример реализации системы автоматической классификации судебных документов, интегрирующий все рассмотренные ранее этапы в единый воспроизводимый конвейер. Цель — демонстрация промышленно-близкого подхода к разработке классификатора, включающего подбор гиперпараметров, сравнение моделей и итоговое оценивание качества.

Задача формулируется следующим образом: дана коллекция судебных документов $\mathcal{D} = \{(d_i, y_i)\}_{i=1}^{N}$, где $y_i \in \{\text{гражданское}, \text{уголовное}, \text{административное}, \text{арбитражное}\}$. Требуется построить классификатор $f: D \to \mathcal{Y}$, максимизирующий $F_1$-меру с взвешиванием по классам на тестовой выборке.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
)
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
)
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.calibration import CalibratedClassifierCV

# ══════════════════════════════════════════════════════════════
# Шаг 1. Формирование расширенного датасета
# ══════════════════════════════════════════════════════════════
extended_templates = {
    'гражданское': [
        "взыскание задолженности договор поставка товар неоплата",
        "расторжение договора аренда помещение невыполнение обязательств",
        "алименты содержание ребёнок развод взыскание",
        "признание права собственность имущество наследство",
        "возмещение ущерб ДТП страховая компания отказ выплата",
        "трудовой спор незаконное увольнение восстановление работа",
    ],
    'уголовное': [
        "кража проникновение жилище срок заключение условно",
        "мошенничество хищение денежные средства обман доверие",
        "хулиганство общественный порядок нарушение штраф",
        "наркотики хранение распространение статья лишение свободы",
        "убийство умышленное причинение тяжкий вред здоровью",
        "оправдательный приговор отсутствие доказательств вина",
    ],
    'административное': [
        "штраф превышение скорость камера ГИБДД",
        "нарушение пожарная безопасность предписание устранение",
        "лицензия отзыв деятельность прекращение нарушение условий",
        "предупреждение санитарные нормы торговля продукты",
        "депортация нарушение режим пребывание иностранный гражданин",
        "арест административный сутки нарушение режим",
    ],
    'арбитражное': [
        "банкротство несостоятельность конкурсное производство должник кредитор",
        "налоговый спор недоимка пени штраф оспаривание решение",
        "корпоративный спор акционер общество обжалование решение",
        "мировое соглашение стороны урегулирование спор",
        "контракт государственный заказ неисполнение неустойка",
        "интеллектуальная собственность товарный знак нарушение использование",
    ],
}

texts_ext, labels_ext = [], []
np.random.seed(0)
for label, tmpl_list in extended_templates.items():
    for _ in range(75):
        t = np.random.choice(tmpl_list)
        extra = np.random.choice([
            "рассмотрев обстоятельства дела суд установил",
            "изучив представленные доказательства суд пришёл выводу",
            "суд удовлетворил отказал исковые требования",
        ])
        texts_ext.append(f"{t} {extra}")
        labels_ext.append(label)

X_tr, X_te, y_tr, y_te = train_test_split(
    texts_ext, labels_ext, test_size=0.2,
    stratify=labels_ext, random_state=42
)

print(f"Обучение: {len(X_tr)} | Тест: {len(X_te)}")
print("Распределение классов в обучающей выборке:")
for cls, cnt in sorted(pd.Series(y_tr).value_counts().items()):
    print(f"  {cls:<20}: {cnt}")

# ══════════════════════════════════════════════════════════════
# Шаг 2. Подбор гиперпараметров (GridSearchCV)
# ══════════════════════════════════════════════════════════════
base_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf',   LogisticRegression(max_iter=1000, random_state=42)),
])

param_grid = {
    'tfidf__max_features':  [3000, 5000],
    'tfidf__ngram_range':   [(1,1), (1,2)],
    'tfidf__sublinear_tf':  [True],
    'clf__C':               [0.5, 1.0, 5.0],
}

cv_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    base_pipeline, param_grid,
    cv=cv_inner, scoring='f1_weighted',
    n_jobs=-1, verbose=0, refit=True,
)
grid_search.fit(X_tr, y_tr)

print(f"\nЛучшие гиперпараметры: {grid_search.best_params_}")
print(f"Лучший CV F₁: {grid_search.best_score_:.4f}")

# ══════════════════════════════════════════════════════════════
# Шаг 3. Финальная оценка на тестовой выборке
# ══════════════════════════════════════════════════════════════
best_model = grid_search.best_estimator_
y_pred_final = best_model.predict(X_te)

acc  = accuracy_score(y_te, y_pred_final)
f1w  = f1_score(y_te, y_pred_final, average='weighted')
f1m  = f1_score(y_te, y_pred_final, average='macro')

print(f"\n=== Результаты на тестовой выборке ===")
print(f"Accuracy:      {acc:.4f}")
print(f"F₁-weighted:   {f1w:.4f}")
print(f"F₁-macro:      {f1m:.4f}")
print("\nДетальный отчёт:")
print(classification_report(y_te, y_pred_final, digits=4))

# ══════════════════════════════════════════════════════════════
# Шаг 4. Визуализация матрицы ошибок
# ══════════════════════════════════════════════════════════════
cm = confusion_matrix(y_te, y_pred_final, labels=sorted(set(labels_ext)))
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=sorted(set(labels_ext)))
disp.plot(ax=ax, colorbar=True, cmap='Blues', values_format='d')
ax.set(title=f'Матрица ошибок (Accuracy={acc:.4f}, F₁={f1w:.4f})')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/final_confusion_matrix.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Матрица ошибок сохранена.")


---

### 12. Этические аспекты и ограничения автоматизации правового анализа

#### 12.1. Предвзятость алгоритмических систем в правовой сфере

Автоматизация классификации и анализа судебных документов открывает значительные возможности для повышения эффективности правосудия, однако сопровождается рядом принципиальных этических рисков, требующих осознанного рассмотрения со стороны как разработчиков, так и правоприменителей.

Фундаментальной проблемой является предвзятость (bias) обученных моделей, воспроизводящих исторические закономерности судебных решений, которые могут включать систематические дискриминационные паттерны по признакам пола, возраста, национальности или социального статуса. Исследования алгоритма COMPAS (Correctional Offender Management Profiling for Alternative Sanctions), применявшегося в США для оценки риска рецидива, показали, что система демонстрировала значимо более высокие прогнозы риска для афроамериканцев по сравнению с белыми при сопоставимых обстоятельствах дела. Механическое применение подобных систем без критического осмысления способно институционализировать и усилить существующее неравенство.

В российском контексте риск воспроизведения предвзятости релевантен в контексте региональной неоднородности судебной практики: модель, обученная преимущественно на данных столичных судов, может систематически ошибаться при работе с документами региональных инстанций.

#### 12.2. Вопросы интерпретируемости и объяснимости

С позиции требований правового государства (rule of law) особое значение приобретает принцип объяснимости (explainability) алгоритмических решений. Метод $\text{LIME}$ (Local Interpretable Model-agnostic Explanations) и метод SHAP (SHapley Additive exPlanations) позволяют получить локальные объяснения предсказаний «чёрного ящика» для конкретного документа.

Значение Шепли для признака $j$ в предсказании модели $f$ для объекта $x$ вычисляется как средневзвешенный вклад признака $j$ по всем подмножествам признаков:

$$\phi_j(x) = \sum_{S \subseteq F \setminus \{j\}} \frac{|S|!(|F| - |S| - 1)!}{|F|!} \left[f(S \cup \{j\}) - f(S)\right]$$

где $F$ — множество всех признаков, $f(S)$ — ожидаемое предсказание модели при использовании только признаков из подмножества $S$.

#### 12.3. Правовые ограничения и принципы ответственного внедрения

Применение автоматизированных систем принятия решений в правовой сфере регулируется рядом нормативных документов. В частности, Регламент ЕС об искусственном интеллекте (AI Act, 2024) классифицирует системы, применяемые в сфере правосудия, как высокорисковые, что влечёт обязанность проведения оценки воздействия на основные права и обеспечения человеческого контроля за автоматизированными решениями.

Принципы ответственного внедрения систем автоматического анализа судебных документов включают обеспечение прозрачности и объяснимости алгоритмических решений, регулярный аудит моделей на предмет предвзятости, сохранение за человеком права окончательного решения, а также документирование ограничений и условий применимости системы.

---


### Заключение

Настоящая лекция представила системное изложение методов автоматизации классификации и анализа судебных документов — от теоретических основ машинного обучения до практических аспектов разработки и оценки классификационных систем.

В ходе изложения была рассмотрена полная цепочка обработки: начиная с автоматизированного сбора текстовых данных и их нормализации посредством морфологического анализа, через методы векторного представления текстов (Bag of Words, TF-IDF, Word Embeddings) к алгоритмам классификации — от логистической регрессии и метода опорных векторов до рекуррентных нейронных сетей LSTM и трансформерных архитектур. Отдельное внимание уделено задаче обнаружения аномалий (Isolation Forest, One-Class SVM) как инструменту выявления нетипичных судебных документов.

Принципиально важным является понимание того, что качество системы не исчерпывается единственной числовой метрикой: для задач юридической аналитики необходимо детальное исследование матрицы ошибок, анализ Per-class метрик и учёт асимметрии цены различных типов ошибок. Кросс-валидация обеспечивает надёжные оценки обобщающей способности модели.

Наконец, этические аспекты автоматизации правового анализа — предвзятость алгоритмов, необходимость объяснимости, правовые ограничения — должны рассматриваться не как второстепенные, а как неотъемлемая часть проектирования любой системы, претендующей на применение в юридической практике.

---

**Вопросы для самоконтроля:**

1. Чем принципиально отличается метрика TF-IDF от модели «мешок слов» с точки зрения математической формализации? При каких условиях IDF-компонента обращается в нуль?
2. Почему классические RNN страдают от проблемы исчезающего градиента? Какие механизмы LSTM решают эту проблему?
3. Сформулируйте задачу One-Class SVM в виде задачи оптимизации. Как параметр $\nu$ влияет на характеристики обнаружения аномалий?
4. Объясните, почему несбалансированность классов делает метрику Accuracy неадекватной мерой качества. Какие альтернативы следует использовать?
5. В чём состоит риск предвзятости обученных классификаторов судебных документов? Предложите методологию выявления и митигации предвзятости.

**Задание для самостоятельной работы:**

Реализуйте полный конвейер классификации судебных документов по следующей схеме. На открытом корпусе судебной практики (например, подмножестве датасета RUSLAN или данных «ГАС Правосудие») проведите: (1) предобработку с использованием pymorphy2; (2) сравнение TF-IDF и Word2Vec представлений; (3) оценку трёх классификаторов (LogisticRegression, LinearSVC, LSTM) с использованием стратифицированной 5-кратной кросс-валидации; (4) визуализацию результатов через t-SNE и матрицу ошибок; (5) интерпретацию наиболее влиятельных признаков методом SHAP. Результаты оформите в виде Jupyter Notebook с академическими комментариями.

---

**Список рекомендуемой литературы:**

- Jurafsky, D., & Martin, J. H. (2023). *Speech and Language Processing* (3rd ed., draft). Stanford University Press.
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press.
- Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation*, 9(8), 1735–1780.
- Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2019). BERT: Pre-training of deep bidirectional transformers for language understanding. *Proceedings of NAACL-HLT 2019*, 4171–4186.
- Liu, F. T., Ting, K. M., & Zhou, Z.-H. (2008). Isolation forest. *Proceedings of ICDM 2008*, 413–422.
- Предко, О. А., Алдарова, И. А. (2022). Применение методов машинного обучения в автоматизации анализа судебных документов. *Прикладная информатика*, 17(3), 45–62.
